In [ ]:
class AcademicAdvisorAgent:
    def __init__(self):
        # KB - knowledge base
        self.KB = {
            "weights": {
                "attendance": 0.3,
                "coursework": 0.4,
                "deadline": 0.2,
                "difficulty": 0.1
            },

            # Prior (assujmed) probabilities (before seeing evidence)
            "priors": {
                "high_risk": 0.3,
                "medium_risk": 0.4,
                "low_risk": 0.3
            },

            # Rules
            "rules": [
                "low_attendance -> higher_risk",
                "low_coursework -> higher_risk",
                "deadline_close -> higher_risk",
                "high_performance -> lower_risk"
            ]
        }
    # def __init__(self):
    #     # wegiths represent importance of each factor
    #     self.weights = {
    #         "attendance": 0.3,
    #         "coursework": 0.4,
    #         "deadline": 0.2,
    #         "difficulty": 0.1
    #     }

    def interpret_input(self, student_data):
        return student_data
    
    # inference (weighted)
    def infer_risk(self, data):
        # this function calculates how risky the student's situation is
        # takes student data -> calculates risk score -> decides risk level -> explains why -> gives confidence

        explanation = [] # stores reasons 
        score = 0 # total risk amount
        total_weight = 0 # how much data we used

        # attendance
        if "attendance" in data:
            att = data["attendance"]
            if att < 50:
                score += self.KB["weights"]["attendance"] * 1.0
                explanation.append("Very low attendance")
            elif att < 70:
                score += self.KB["weights"]["attendance"] * 0.5
                explanation.append("Moderate attendance")
            else:
                explanation.append("Good attendance")
            total_weight += self.KB["weights"]["attendance"]


        # coursework
        if "coursework" in data:
            cw = data["coursework"]
            if cw < 50:
                score += self.KB["weights"]["coursework"] * 1.0
                explanation.append("Low coursework score")

            elif cw < 65:
                score += self.KB["weights"]["coursework"] * 0.5
                explanation.append("Average coursework")
            else:
                explanation.append("Strong coursework performance")
            total_weight += self.KB["weights"]["coursework"]

        
        # deadline proximity
        if "deadline" in data:
            d = data["deadline"]
            if d <= 3:
                score += self.KB["weights"]["deadline"] * 1.0
                explanation.append("Deadline very close")
            elif d <= 7:
                score += self.KB["weights"]["deadline"] * 0.5
                explanation.append("Deadline approaching")
            total_weight += self.KB["weights"]["deadline"]


        # difficulty perception
        if "difficulty" in data:
            if data["difficulty"] == "hard":
                score += self.KB["weights"]["difficulty"] * 1.0
                explanation.append("Subject perceived as difficult")
            elif data["difficulty"] == "medium":
                score += self.KB["weights"]["difficulty"] * 0.5
                explanation.append("Moderate difficulty")
            total_weight += self.KB["weights"]["difficulty"]

        # normalize score (0 to 1)
        # convert score into a value between 0 and 1
        if total_weight == 0:
            return "unknown", 0.3, ["No useful data provided"]
        
        risk_score = score / total_weight

        # risk classification
        if risk_score > 0.7:
            risk = "high_risk"
        elif risk_score > 0.4:
            risk = "medium_risk"
        else:
            risk = "low_risk"

        # confidence = how much data we used 
        confidence = total_weight # more data = higher confidence
        return risk, round(confidence, 2), explanation



    # decision logic
    def decide_action(self, risk_level, confidence):
        # takes the risk level and returns the advice message

        if confidence < 0.5:
            return "I am not confident due to limited information. Please provide more details."
        

        if risk_level == "high_risk":
            return "You are at high risk. Prioritise urgent revision and seek academic support"
        elif risk_level == "medium_risk":
            return "You are at moderate risk. Improve consistency and focus on weak areas"
        elif risk_level == "low_risk":
            return "You are performing well. Maintain your study strategy"
        else:
            return "I need more information to assess your sitation"


    # explainability
    def explain(self, explanation, confidence, risk):
        return (
            f"Risk level: {risk}\n"
            f"Reasoning: {', '.join(explanation)}\n"
            f"Based on rules: {', '.join(self.KB['rules'])}\n"
            f"Confidence: {confidence}"
        )
    
    # bayesian risk estimate function
    def bayesian_risk_estimate(self, data):
        # estimate probabilities for each risk level
        # how likely is each risk level given the student data
        # update belief when you see evidence
        
        # # assumed prior probabilities
        # p_high = 0.3
        # p_medium = 0.4
        # p_low = 0.3

        p_high = self.KB["priors"]["high_risk"]
        p_medium = self.KB["priors"]["medium_risk"]
        p_low = self.KB["priors"]["low_risk"]
        # heuristics:
        # 1.5 - strong signal
        # 1.4 - medium signal
        # 1.2 - weaker signal

        # update probabilities based on evidence
        # attendance effect
        if "attendance" in data:
            att = data["attendance"]
            if att < 50:
                p_high *= 1.5 # low attendance => high risk (more likely)
                p_medium *= 1.2
            elif att > 75:
                p_low *= 1.5

        # coursework effect
        if "coursework" in data:
            cw = data["coursework"]
            if cw < 50:
                p_high *= 1.5
            elif cw > 70:
                p_low *= 1.5

        # deadline effect
        if "deadline" in data:
            d = data["deadline"]
            if d <= 3:
                p_high *= 1.4
            elif d > 7:
                p_low *= 1.2

        # normalize probabilities
        # We divide each value by the total so all probabilities sum to 1
        total = p_high + p_medium + p_low

        p_high /= total
        p_medium /= total
        p_low /= total

        return {
            "high_risk": round(p_high, 2),
            "medium_risk": round(p_medium, 2),
            "low_risk": round(p_low, 2)
        }
    
    # full pipeline
    def run(self, student_data):
        data = self.interpret_input(student_data)
        risk, confidence, explanation = self.infer_risk(data)

        # bayesian probabilities
        probabilities = self.bayesian_risk_estimate(data)

        decision = self.decide_action(risk, confidence)
        explanation_text = self.explain(explanation, confidence, risk)

        return {
            "decision": decision,
            "explanation": explanation_text,
            "probabilities": probabilities
        }
    

    # get user input function
    def get_user_input(self):
        data = {}
        print("Answer the following questions (press Enter to skip):")

        # attendance
        att = input("What is your attendance percentage? ")
        if att:
            data["attendance"] = int(att)

        # coursework
        cw = input("What is your coursework score? ")
        if cw:
            data["coursework "] = int(cw)

        # deadline
        d = input("Days until next deadline? ")
        if d:
            data["deadline"] = int(d)

        # difficulty
        diff = input("How difficult is the subject? (easy/medium/hard): ")
        if diff:
            data["difficulty"] = diff.lower()

        return data
    

    def interpret_user_message(self, message):
        message = message.lower()
        data = {}

        # detect intent
        if "struggle" in message or "risk" in message:
            intent = "risk_assessment"
        else:
            intent = "general"

        # extract numbers 
        words = message.split()

        for w in words:
            if w.isdigit():
                val = int(w)
                if val <= 100:
                    # naive mapping
                    if "attendance" in message:
                        data["attendance"] = val
                    elif "coursework" in message:
                        data["coursework"] = val

        # detect deadline
        if "day" in message:
            for w in words:
                if w.isdigit():
                    data["deadline"] = int(w)

        # detect difficulty
        if "hard" in message:
            data["difficulty"] = "hard"
        elif "medium" in message:
            data["difficulty"] = "medium"
        elif "easy" in message:
            data["difficulty"] = "easy"

        return intent, data
    

    def ask_missing_info(self, data):
        if "attendance" not in data:
            val = input("What is your attendance percentage? ")
            if val:
                data["attendance"] = int(val)

        if "coursework" not in data:
            val = input("What is your coursework score? ")
            if val:
                data["coursework"] = int(val)

        if "deadline" not in data:
            val = input("Days until next deadline? ")
            if val:
                data["deadline"] = int(val)

        if "difficulty" not in data:
            val = input("How difficult is the subject? (easy/medium/hard): ")
            if val:
                data["difficulty"] = val.lower()

        return data
    


    def run_smart_agent(self):
        print("Academic Advisor Agent (type 'exit' to quit)\n")

        while True:
            message = input("You: ")

            if message.lower() == "exit":
                print("Goodbye.")
                break

            intent, data = self.interpret_user_message(message)

            if intent != "risk_assessment":
                print("Agent: I can help assess your academic risk. Try describing your situation.")
                continue

            # ask only missing info
            data = self.ask_missing_info(data)

            result = self.run(data)

            print("\nAgent:", result["decision"])
            print("Explanation:", result["explanation"])
            print("Probabilities:", result["probabilities"])
            print("-" * 50)
    
    # conversation loop
    def run_interactive(self):
        print("Academic Advisor Agent (type 'exit' to quit)\n")

        while True:
            command = input("Type 'start' to get advice or 'exit': ").lower()

            if command == "exit":
                print("Goodbye.")
                break

            if command == "start":
                # get the student data
                student_data = self.get_user_input()
                
                # run the program with the given data
                result = self.run(student_data)

                
                print("\nDecision:")
                print(result["decision"])

                print("\nExplanation:")
                print(result["explanation"])

                print("\nProbabilities:")
                print(result["probabilities"])

                print()
                print("-" * 50)

    


In [ ]:
agent = AcademicAdvisorAgent()
agent.run_interactive()